# Week 5 - Apache Spark Basics
**Dataset:** Sample Superstore (9,994 retail orders)

Goal for this notebook: load the Superstore data into a Spark DataFrame, clean it up, filter and transform it, then run some aggregations and grouped summaries, and finally wrap the whole thing into one pipeline function.

## Step 1-2: Why Spark, and starting a Spark session

MapReduce reads/writes to disk between every step, which is slow. Spark keeps data in memory across operations and gives you DataFrames to work with, so it's both faster and easier to write than raw MapReduce jobs.

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, count, avg, min, max, sum, year, to_date

spark = SparkSession.builder \
    .appName("SuperstoreSparkBasics") \
    .master("local[*]") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")
print("Spark session started. Version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/21 14:38:53 WARN Utils: Your hostname, vm, resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/06/21 14:38:53 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/06/21 14:38:54 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark session started. Version: 4.1.2


## Step 3: Load Data

Note on `multiLine=True` below: without it, Spark's CSV parser mis-handles 300 `Product Name` values (about 3% of the dataset) that contain escaped inch-mark quotes (like `14 7/8""`). That mismatch silently shifts `Sales`, `Quantity`, and `Discount` into the wrong columns and Spark infers them as strings instead of numbers. Found this the hard way when a later `.filter(col("Sales") > 100)` crashed with a cast error - more on this below.

In [2]:
df = spark.read.option("multiLine", True).option("quote", '"').option("escape", '"') \
    .csv("../data/dataset.csv", header=True, inferSchema=True)

print("First 5 rows:")
df.show(5)

First 5 rows:


+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|Row ID|      Order ID|Order Date| Ship Date|     Ship Mode|Customer ID|  Customer Name|  Segment|      Country|           City|     State|Postal Code|Region|     Product ID|       Category|Sub-Category|        Product Name|   Sales|Quantity|Discount|  Profit|
+------+--------------+----------+----------+--------------+-----------+---------------+---------+-------------+---------------+----------+-----------+------+---------------+---------------+------------+--------------------+--------+--------+--------+--------+
|     1|CA-2016-152156| 11/8/2016|11/11/2016|  Second Class|   CG-12520|    Claire Gute| Consumer|United States|      Henderson|  Kentucky|      42420| South|FUR-BO-10001798|      Furniture|   Bookcases|Bush Somerset 

In [3]:
print("Column names:", df.columns)

Column names: ['Row ID', 'Order ID', 'Order Date', 'Ship Date', 'Ship Mode', 'Customer ID', 'Customer Name', 'Segment', 'Country', 'City', 'State', 'Postal Code', 'Region', 'Product ID', 'Category', 'Sub-Category', 'Product Name', 'Sales', 'Quantity', 'Discount', 'Profit']


In [4]:
print("Schema / data types:")
df.printSchema()
print("Total rows loaded:", df.count())

Schema / data types:
root
 |-- Row ID: integer (nullable = true)
 |-- Order ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: integer (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub-Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)



Total rows loaded: 9994


## Step 4: Data Cleaning

Checking for duplicate rows and nulls, then confirming the `multiLine` fix actually worked on the numeric columns.

In [5]:
before = df.count()
df_clean = df.dropDuplicates()
after = df_clean.count()
print(f"Duplicate rows removed: {before - after}")

Duplicate rows removed: 0


In [6]:
print("Null count per column:")
null_counts = df_clean.select([
    sum(col(c).isNull().cast("int")).alias(c) for c in df_clean.columns
])
null_counts.show()

# Zero nulls here, which is a bit unusual - most real datasets aren't this tidy,
# but it's still worth running the check rather than assuming.
df_clean = df_clean.na.drop()

Null count per column:


+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|Row ID|Order ID|Order Date|Ship Date|Ship Mode|Customer ID|Customer Name|Segment|Country|City|State|Postal Code|Region|Product ID|Category|Sub-Category|Product Name|Sales|Quantity|Discount|Profit|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+
|     0|       0|         0|        0|        0|          0|            0|      0|      0|   0|    0|          0|     0|         0|       0|           0|           0|    0|       0|       0|     0|
+------+--------+----------+---------+---------+-----------+-------------+-------+-------+----+-----+-----------+------+----------+--------+------------+------------+-----+--------+--------+------+



In [7]:
print("Sanity check - Sales/Quantity/Discount types after the multiLine fix:")
df_clean.select("Sales", "Quantity", "Discount").printSchema()

Sanity check - Sales/Quantity/Discount types after the multiLine fix:
root
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)



## Step 5: Filter Data

The original assignment template filters by age, but this is retail order data with no age column. Used `Sales` as the equivalent numeric range filter instead, combined with `Category` and `Region`.

In [8]:
filtered_df = df_clean.filter(
    (col("Category") == "Furniture") &
    (col("Region") == "West") &
    (col("Sales") > 100)
)
print(f"Rows after filtering (Furniture, West region, Sales > 100): {filtered_df.count()}")
filtered_df.select("Order ID", "Category", "Region", "Sales", "Profit").show(10)

Rows after filtering (Furniture, West region, Sales > 100): 458


+--------------+---------+------+--------+---------+
|      Order ID| Category|Region|   Sales|   Profit|
+--------------+---------+------+--------+---------+
|CA-2017-142636|Furniture|  West| 307.136|  26.8744|
|CA-2014-135657|Furniture|  West|  515.88| 113.4936|
|CA-2016-146913|Furniture|  West| 1403.92|   70.196|
|CA-2014-116932|Furniture|  West| 272.848|  27.2848|
|US-2015-165743|Furniture|  West| 145.764|-247.7988|
|US-2017-105998|Furniture|  West|1673.184|  20.9148|
|CA-2015-114237|Furniture|  West|  141.96|  39.7488|
|CA-2014-111871|Furniture|  West| 1198.33|    70.49|
|US-2017-109316|Furniture|  West|1497.666| 140.9568|
|CA-2016-167241|Furniture|  West|  312.03|  43.6842|
+--------------+---------+------+--------+---------+
only showing top 10 rows


## Step 6: Transform Data

A few small fixes here:
- Renamed columns that had spaces, so they're easier to reference in code
- `Postal Code` was auto-inferred as an integer, but it's an identifier, not something you'd ever do math on, so it's cast to a string
- Parsed `Order Date` into a real date type and pulled out the year, for grouping later

In [9]:
transformed_df = df_clean \
    .withColumnRenamed("Order ID", "Order_ID") \
    .withColumnRenamed("Customer Name", "Customer_Name") \
    .withColumnRenamed("Sub-Category", "Sub_Category")

transformed_df = transformed_df.withColumn("Postal Code", col("Postal Code").cast("string"))

transformed_df = transformed_df.withColumn("Order_Date_Parsed", to_date(col("Order Date"), "M/d/yyyy"))
transformed_df = transformed_df.withColumn("Order_Year", year(col("Order_Date_Parsed")))

print("Schema after transformation:")
transformed_df.printSchema()

Schema after transformation:
root
 |-- Row ID: integer (nullable = true)
 |-- Order_ID: string (nullable = true)
 |-- Order Date: string (nullable = true)
 |-- Ship Date: string (nullable = true)
 |-- Ship Mode: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Customer_Name: string (nullable = true)
 |-- Segment: string (nullable = true)
 |-- Country: string (nullable = true)
 |-- City: string (nullable = true)
 |-- State: string (nullable = true)
 |-- Postal Code: string (nullable = true)
 |-- Region: string (nullable = true)
 |-- Product ID: string (nullable = true)
 |-- Category: string (nullable = true)
 |-- Sub_Category: string (nullable = true)
 |-- Product Name: string (nullable = true)
 |-- Sales: double (nullable = true)
 |-- Quantity: integer (nullable = true)
 |-- Discount: double (nullable = true)
 |-- Profit: double (nullable = true)
 |-- Order_Date_Parsed: date (nullable = true)
 |-- Order_Year: integer (nullable = true)



## Step 7: Aggregation

Basic stats across the whole dataset.

In [10]:
transformed_df.select(
    count("*").alias("total_orders"),
    avg("Sales").alias("avg_sales"),
    min("Sales").alias("min_sales"),
    max("Sales").alias("max_sales"),
    avg("Profit").alias("avg_profit"),
    min("Profit").alias("min_profit"),
    max("Profit").alias("max_profit")
).show()

+------------+-----------------+---------+---------+------------------+----------+----------+
|total_orders|        avg_sales|min_sales|max_sales|        avg_profit|min_profit|max_profit|
+------------+-----------------+---------+---------+------------------+----------+----------+
|        9994|229.8580008304938|    0.444| 22638.48|28.656896307784624| -6599.978|  8399.976|
+------------+-----------------+---------+---------+------------------+----------+----------+



## Step 8: Group Data

In [11]:
print("Sales and profit grouped by Region:")
region_summary = transformed_df.groupBy("Region") \
    .agg(
        count("*").alias("order_count"),
        sum("Sales").alias("total_sales"),
        avg("Profit").alias("avg_profit")
    ) \
    .orderBy(col("total_sales").desc())
region_summary.show()

Sales and profit grouped by Region:


+-------+-----------+------------------+------------------+
| Region|order_count|       total_sales|        avg_profit|
+-------+-----------+------------------+------------------+
|   West|       3203| 725457.8244999993| 33.84903181392447|
|   East|       2848| 678781.2399999988| 32.13580758426972|
|Central|       2323|501239.89080000005|17.092708781747756|
|  South|       1620|391721.90500000055|28.857673024691355|
+-------+-----------+------------------+------------------+



In [12]:
print("Sales grouped by Category and Sub_Category:")
category_summary = transformed_df.groupBy("Category", "Sub_Category") \
    .agg(
        count("*").alias("order_count"),
        sum("Sales").alias("total_sales"),
        avg("Profit").alias("avg_profit")
    ) \
    .orderBy(col("total_sales").desc())
category_summary.show(15)

Sales grouped by Category and Sub_Category:


+---------------+------------+-----------+------------------+-------------------+
|       Category|Sub_Category|order_count|       total_sales|         avg_profit|
+---------------+------------+-----------+------------------+-------------------+
|     Technology|      Phones|        889| 330007.0540000002| 50.073937682789605|
|      Furniture|      Chairs|        617| 328449.1030000004| 43.095893517017856|
|Office Supplies|     Storage|        846|        223843.608| 25.152277068557943|
|      Furniture|      Tables|        319|206965.53200000004| -55.56577147335426|
|Office Supplies|     Binders|       1523|203412.73300000012| 19.843574064346722|
|     Technology|    Machines|        115|189238.63100000005|  29.43266869565216|
|     Technology| Accessories|        775|167380.31800000006|  54.11178799999999|
|     Technology|     Copiers|         68| 149528.0299999999|  817.9091897058823|
|      Furniture|   Bookcases|        228|114879.99629999998|-15.230508771929825|
|Office Supplies

## Step 9: Wide transformations and shuffle (just the basic idea)

`groupBy()` and `orderBy()` above are both **wide transformations**. To compute them, Spark has to gather rows that share a key (e.g. all "West" region rows) from across however many partitions the data is split into, onto the same worker, before it can sum/average/sort anything. That movement of data between partitions is the **shuffle**, and it's the expensive part of a Spark job.

By contrast, `filter()` and `withColumn()` from earlier are **narrow transformations** - each partition handles its own rows independently, no data needs to move around, so they're much cheaper.

## Step 10: Build a Simple Pipeline

Wrapping load → clean → filter → transform → aggregate into one function, so the whole thing can be re-run end to end on the raw file.

In [13]:
def run_pipeline(path):
    raw = spark.read.option("multiLine", True).option("quote", '"').option("escape", '"') \
        .csv(path, header=True, inferSchema=True)
    cleaned = raw.dropDuplicates().na.drop()
    filtered = cleaned.filter(col("Sales") > 50)
    transformed = filtered.withColumnRenamed("Order ID", "Order_ID") \
                           .withColumn("Postal Code", col("Postal Code").cast("string"))
    result = transformed.groupBy("Region", "Category") \
        .agg(
            count("*").alias("order_count"),
            sum("Sales").alias("total_sales"),
            avg("Profit").alias("avg_profit")
        ) \
        .orderBy(col("total_sales").desc())
    return result

pipeline_result = run_pipeline("../data/dataset.csv")
print("Final pipeline output (Region x Category summary, Sales > 50):")
pipeline_result.show(50)

Final pipeline output (Region x Category summary, Sales > 50):


+-------+---------------+-----------+------------------+------------------+
| Region|       Category|order_count|       total_sales|        avg_profit|
+-------+---------------+-----------+------------------+------------------+
|   East|     Technology|        429|261917.89900000015|109.96219300699298|
|   West|     Technology|        502|249096.30299999996| 87.45234462151399|
|   West|      Furniture|        533|248280.21250000046|19.173459662288906|
|   East|      Furniture|        456|204703.02400000003| 4.545038157894747|
|   West|Office Supplies|        683|197210.10400000014| 66.07031932650075|
|   East|Office Supplies|        586| 184466.7820000001| 60.61112508532419|
|Central|     Technology|        330|        167931.254| 101.1527754545455|
|Central|      Furniture|        343|161170.59379999994|-6.110033527696797|
|Central|Office Supplies|        452|150509.99800000008| 21.17279092920357|
|  South|     Technology|        232|        146974.179| 84.78192844827589|
|  South|   

## Saving the output

In [14]:
pipeline_result.toPandas().to_csv("../output/results.csv", index=False)
print("Saved pipeline_result to ../output/results.csv")

Saved pipeline_result to ../output/results.csv


In [15]:
spark.stop()